In [1]:
# Cell A: Install dependencies
# rank_bm25 + sentence-transformers add hybrid RAG
# nest_asyncio enables asyncio.run() inside Colab's running event loop
!pip install langgraph langchain-openai langchain pydantic pandas scikit-learn \
             python-docx plotly gradio textblob rank_bm25 sentence-transformers \
             nest_asyncio -q
!python -m textblob.download_corpora
!pip install langgraph langchain-openai langchain pydantic pandas scikit-learn \
             python-docx plotly gradio textblob rank_bm25 sentence-transformers \
             nest_asyncio kaleido -q
!pip install --upgrade gradio huggingface_hub

print("Dependencies installed.")

[nltk_data] Downloading package brown to /root/nltk_data...
[nltk_data]   Package brown is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package conll2000 to /root/nltk_data...
[nltk_data]   Package conll2000 is already up-to-date!
[nltk_data] Downloading package movie_reviews to /root/nltk_data...
[nltk_data]   Package movie_reviews is already up-to-date!
Finished.
Dependencies installed.


In [2]:
!pip install -U kaleido==0.2.1

In [3]:
# Cell B: API key + configuration
import os, warnings
warnings.filterwarnings("ignore")

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("API key loaded from Colab secrets.")
except Exception:
    import getpass
    if "OPENAI_API_KEY" not in os.environ:
        os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API key: ")
    print("API key set.")

CONFIG = {
    # Model
    "model": "gpt-4o-mini",
    "temperature": 0.0,

    # Cost / rate control
    "max_reviews": 30,          # per-brand cap for sentiment analysis
    "max_brands": 10,
    "llm_batch_size": 5,
    "llm_max_retries": 3,

    # Retrieval pipeline
    "rerank_candidates": 50,    # cross-encoder sees this many candidates
    "final_top_k": 30,          # reviews returned after re-ranking
    "brand_boost": 0.25,        # RRF weight bonus for queried brands

    # Reflection (Evaluator-Optimizer)
    "reflection_threshold": 7,  # score / 10; below → retry analysis
    "reflection_max_iter": 2,   # bound retries to control cost


    "episodic_file": "episodic_memory.json",
    "brand_mapping_file": "brand_mapping.pkl",
    "compress_min_words": 50,   # reviews shorter than this are kept as-is
    "compress_sentences": 2,    # keep top-N most sentiment-dense sentences

    # Sentiment thresholds
    "pos_thresh": 0.1,
    "neg_thresh": -0.1,

    # Misc
    "use_baseline": True,
    "cache_results": True,
    "verbose": True,
}

print(f"Model: {CONFIG['model']}  |  Max reviews/brand: {CONFIG['max_reviews']}")

API key set.
Model: gpt-4o-mini  |  Max reviews/brand: 30


In [4]:
# Cell C: Imports
import pandas as pd
import numpy as np
import re, json, time, pickle, asyncio
from pathlib import Path
from typing import TypedDict, List, Dict, Optional, Literal, Annotated
from datetime import datetime
from collections import defaultdict
try:
    from google.colab import files
    IN_COLAB = True
except ModuleNotFoundError:
    IN_COLAB = False  # running locally

# LangChain / LangGraph
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langgraph.graph import StateGraph, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import create_react_agent

# Pydantic structured outputs
from pydantic import BaseModel, Field

# ML / NLP
from sklearn.metrics.pairwise import cosine_similarity
from textblob import TextBlob
from rank_bm25 import BM25Okapi                             
from sentence_transformers import SentenceTransformer, CrossEncoder  

# Async inside Colab's running event loop
import nest_asyncio
nest_asyncio.apply()

# Visualization
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

# Report
from docx import Document

# UI
import gradio as gr

print("All imports successful.")

All imports successful.


In [5]:
# Cell D: Data loading + preprocessing + LLM brand normalization

BRAND_NORM_PROMPT = """Standardize these mobile phone brand names to their official forms.

Input brands: {brands}

Rules:
- "Black Berry", "Blackberry", "RIM" -> "BlackBerry"
- "Sony Ericsson" -> "Sony"
- "LG Electronics" -> "LG"
- "Motorola Mobility" -> "Motorola"
- Remove suffixes: Inc, Corp, Ltd, LLC
- Keep unchanged: Apple, Samsung, Nokia, HTC, Google, Huawei, OnePlus, Xiaomi, BLU

Output one line per brand: "Original -> Standard"
"""


def load_brand_mapping(path: str) -> Dict:
    """Load cached brand-name mappings from disk."""
    if Path(path).exists():
        with open(path, "rb") as f:
            return pickle.load(f)
    return {}


def save_brand_mapping(mapping: Dict, path: str):
    with open(path, "wb") as f:
        pickle.dump(mapping, f)


def preprocess_data(df_raw: pd.DataFrame) -> pd.DataFrame:
    """Clean reviews and normalize brand names with LLM (batched + cached)."""
    llm = ChatOpenAI(model=CONFIG["model"], temperature=0)
    df = df_raw.copy()

    # Basic text cleaning
    df["Reviews_Clean"] = df["Reviews"].apply(
        lambda x: re.sub(r"[^\w\s.,!?-]", "", str(x)).strip() if pd.notna(x) else ""
    )
    df["Rating"] = pd.to_numeric(df["Rating"], errors="coerce")
    df["Price"] = pd.to_numeric(df["Price"], errors="coerce")

    # LLM brand normalization: load cache, normalize new brands, save
    mapping = load_brand_mapping(CONFIG["brand_mapping_file"])
    unique_brands = df["Brand Name"].dropna().unique().tolist()
    unmapped = [b for b in unique_brands if b not in mapping]

    for i in range(0, min(len(unmapped), 500), 50):
        batch = unmapped[i : i + 50]
        prompt = BRAND_NORM_PROMPT.format(brands=batch)
        try:
            res = llm.invoke([HumanMessage(prompt)])
            for line in res.content.split("\n"):
                if "->" in line:
                    orig, std = line.split("->", 1)
                    mapping[orig.strip()] = std.strip()
            time.sleep(0.3)
        except Exception as e:
            print(f"  Brand norm batch {i // 50 + 1} failed: {e}")

    save_brand_mapping(mapping, CONFIG["brand_mapping_file"])
    df["Brand_Clean"] = df["Brand Name"].map(lambda x: mapping.get(x, x) if pd.notna(x) else x)

    # Quality filters
    df = df[df["Reviews_Clean"].str.split().str.len() >= 10]
    df = df.drop_duplicates(subset=["Reviews_Clean"])
    df = df.dropna(subset=["Brand_Clean", "Reviews_Clean"]).reset_index(drop=True)

    return df


# Locate dataset file — auto-extract zip if present
import zipfile as _zf

# Locate CSV: check Colab path first, then current directory (local runs)
_colab_csv  = Path("/content/Amazon_Unlocked_Mobile.csv")
_local_csv  = Path("Amazon_Unlocked_Mobile.csv")
_zip_candidates = [
    Path("/content/Amazon_Unlocked_Mobile.csv.zip"),
    Path("Amazon_Unlocked_Mobile.zip"),
    Path("archive.zip"),
]

# Auto-extract zip if neither CSV exists
if not _colab_csv.exists() and not _local_csv.exists():
    for _zip_path in _zip_candidates:
        if _zip_path.exists():
            print(f"Extracting {_zip_path} ...")
            with _zf.ZipFile(_zip_path) as _z:
                _z.extractall("." if not str(_zip_path).startswith("/content") else "/content")
            print("Extraction complete.")
            break

data_path = next((p for p in [_colab_csv, _local_csv] if p.exists()), None)
if data_path is None:
    raise FileNotFoundError(
        "Dataset not found. Place Amazon_Unlocked_Mobile.csv (or its .zip) in the working directory."
    )

print(f"Loading: {data_path}")
df_raw = None # Initialize df_raw here
for _enc in ["utf-8", "utf-8-sig", "latin1", "ISO-8859-1", "cp1252"]:
    try:
        df_raw = pd.read_csv(data_path, encoding=_enc, on_bad_lines="skip", engine="python")
        print(f"Loaded {len(df_raw):,} rows  (encoding={_enc})")
        break
    except Exception as e:
        print(f"  Failed to load with encoding {_enc}: {e}")
        continue

# Add a check to ensure df_raw was successfully loaded
if df_raw is None:
    raise ValueError("Failed to load CSV with any of the specified encodings.")

print("Preprocessing (LLM brand normalization, first run takes a few minutes)...")
df_clean = preprocess_data(df_raw)
print(f"Clean dataset: {len(df_clean):,} reviews  |  {df_clean['Brand_Clean'].nunique()} brands")
print("Top brands:", df_clean["Brand_Clean"].value_counts().head(5).to_dict())

Loading: /content/Amazon_Unlocked_Mobile.csv
Loaded 413,840 rows  (encoding=utf-8)
Preprocessing (LLM brand normalization, first run takes a few minutes)...
Clean dataset: 105,160 reviews  |  275 brands
Top brands: {'Samsung': 23325, 'BLU': 12377, 'LG': 10919, 'Apple': 9229, 'Nokia': 7163}


In [6]:
# Cell E: Prompt templates
# All LLM instructions live here — easy to tune without touching agent logic.

class Prompts:
    # Coordinator: {few_shots} filled at runtime from episodic memory
    # {conversation_ctx} = summary of prior turns
    # {cached_brands} = brands already retrieved in this session
    COORDINATOR = """Parse this product review query.

Query: "{query}"
{few_shots}
{conversation_ctx}

Brands already retrieved in this session (can be reused without a new search):
{cached_brands}

Return JSON:
  intent      : "compare" | "analyze" | "search" | "off_topic"
                Use "off_topic" when the query is NOT about mobile phones, product reviews,
                or brand analysis (e.g. general writing tasks, trivia, unrelated questions).
  brands      : list of ALL brand names relevant to this query (empty for off_topic)
  aspects     : subset of [battery, camera, screen, performance, price, build_quality]
  reuse_brands: subset of brands whose reviews are already cached (can skip search)
  new_brands  : subset of brands that need a fresh search (not in cache)
  reasoning   : one-sentence explanation
"""

    # Sentiment: CoT + aspect breakdown (W9 CoT).
    # {critique_hint} is an empty string on first pass;
    # on retry the reflection agent fills it with brand-specific guidance.
    SENTIMENT = """Analyze this mobile phone review for sentiment and product aspects.

Review: \"{review}\"
{critique_hint}
Think step by step:
1. Which product aspects are mentioned (battery, camera, screen, performance, price, build_quality)?
2. For each mentioned aspect, is the reviewer positive, negative, or neutral?
3. What is the overall sentiment, weighing all aspect signals?

Return JSON:
{{
  "reasoning": "<your step-by-step analysis>",
  "overall_sentiment": "positive" | "negative" | "neutral",
  "overall_polarity": <float -1.0 to 1.0>,
  "confidence": <float 0.0 to 1.0>,
  "aspects": [
    {{"aspect": "<name>", "sentiment": "positive"|"negative"|"neutral", "evidence": "<quote>"}}
  ]
}}

Only include aspects actually mentioned in the review.
"""

    # Reflection: quality evaluation + per-brand critique.
    # brand_critiques drives the three improvements on retry:
    #   issues          -> injected into next-round sentiment prompt
    #   increase_sample -> analysis agent pulls 50% more reviews for that brand
    REFLECTION = """You are a quality reviewer for a business intelligence analysis.

Sentiment analysis results:
{results_summary}

Evaluate on:
1. Coverage : >= 10 reviews per brand?
2. Diversity: >= 2 product aspects covered per brand?
3. Confidence: average confidence >= 0.65?
4. Plausibility: results consistent with known brand reputations?

Return JSON:
{{
  "score": <int 0-10>,
  "pass": <bool, true if score >= {threshold}>,
  "issues": ["<overall issue>", ...],
  "suggestions": ["<overall suggestion>", ...],
  "brand_critiques": [
    {{
      "brand": "<brand name>",
      "issues": ["<specific issue for this brand>", ...],
      "increase_sample": <bool, true if pulling more reviews would help>
    }}
  ]
}}
"""


In [7]:
# Cell F: Pydantic models for structured LLM outputs

class QueryIntent(BaseModel):
    intent: Literal["compare", "analyze", "search", "off_topic"]
    brands: List[str]
    aspects: List[str] = []
    reuse_brands: List[str] = []   # brands already in session cache — skip search
    new_brands: List[str] = []     # brands that need a fresh search
    reasoning: str


class AspectSentiment(BaseModel):
    """Sentiment signal for a single product aspect."""
    aspect: str
    sentiment: Literal["positive", "negative", "neutral"]
    evidence: str


class DetailedSentimentResult(BaseModel):
    """CoT sentiment result with per-aspect breakdown (W9 CoT + W7 metadata)."""
    reasoning: str
    overall_sentiment: Literal["positive", "negative", "neutral"]
    overall_polarity: float = Field(ge=-1.0, le=1.0)
    confidence: float = Field(ge=0.0, le=1.0)
    aspects: List[AspectSentiment] = []


class BrandCritique(BaseModel):
    """Per-brand feedback produced by the reflection agent."""
    brand: str
    issues: List[str]
    increase_sample: bool = False   # True -> analysis agent uses larger review cap


class ReflectionScore(BaseModel):
    """Full output of the reflection quality-check call."""
    score: int = Field(ge=0, le=10)
    pass_: bool = Field(alias="pass")
    issues: List[str] = []
    suggestions: List[str] = []
    brand_critiques: List[BrandCritique] = []

    model_config = {"populate_by_name": True}


class AgentState(TypedDict):
    # Input
    query: str
    # Parsed by coordinator
    intent: str
    target_brands: List[str]
    aspects: List[str]
    # Multi-turn routing: which brands come from cache vs need fresh search
    reuse_brands: List[str]
    new_brands: List[str]
    # Search results: brand -> List[Dict] (SERIALIZATION FIX — no DataFrames in state)
    # DataFrames live in _session_reviews (module-level) and REVIEW_STORE (global).
    # Agents read DataFrames from those globals; state only carries serializable records.
    product_reviews: Dict[str, List[Dict]]
    # Analysis results
    sentiment_results: List[Dict]
    aspect_summary: Dict[str, Dict]
    comparison_table: Optional[List[Dict]]   # serializable records, not pd.DataFrame
    # Reflection outputs
    reflection_score: Optional[int]
    reflection_issues: List[str]
    reflection_brand_issues: Dict[str, str]   # brand -> critique hint string for prompt injection
    reflection_increase_sample: List[str]     # brands that need a larger review sample
    reflection_iter: int
    # Multi-turn conversation history
    conversation_history: List[Dict]
    # Output
    viz_charts: Dict
    chart_commentary: Dict
    report_path: Optional[str]
    # Diagnostics
    retrieval_metrics: Dict
    efficiency_metrics: Dict
    messages: Annotated[List, add_messages]
    guidance_message: Optional[str]   # set by coordinator for off-topic / unrecognised queries
    error: Optional[str]
    _search_notes: Dict[str, str]   # set by react_search_agent for skipped brands
    # Internal (_df kept for search agent init only — never serialized by LangGraph)
    _df: Optional[object]
    _start_time: Optional[float]

print("Pydantic models and AgentState defined.")


Pydantic models and AgentState defined.


In [8]:
# Cell G: Episodic Memory + Context Compression + Session Cache

class EpisodicMemory:
    """Stores past query-intent records as few-shot examples for the coordinator.

    W10 concept: episodic memory = "what happened" — past interactions that
    the agent can learn from. Retrieved by Jaccard word overlap (no extra API call).
    Written after every successful query completion.
    """

    def __init__(self, path: str, max_size: int = 20):
        self.path = Path(path)
        self.max_size = max_size
        self.episodes: List[Dict] = self._load()

    def _load(self) -> List[Dict]:
        if self.path.exists():
            with open(self.path) as f:
                return json.load(f)
        return []

    def _save(self):
        with open(self.path, "w") as f:
            json.dump(self.episodes[-self.max_size :], f, indent=2)

    def store(self, query: str, intent: str, brands: List[str], aspects: List[str]):
        """Persist a successful query resolution."""
        self.episodes.append({
            "query": query, "intent": intent,
            "brands": brands, "aspects": aspects,
            "ts": datetime.now().isoformat(),
        })
        self._save()

    def retrieve_similar(self, query: str, k: int = 2) -> List[Dict]:
        """Return up to k episodes most similar to query by Jaccard overlap."""
        if not self.episodes:
            return []
        q_words = set(query.lower().split())
        scored = []
        for ep in self.episodes:
            ep_words = set(ep["query"].lower().split())
            j = len(q_words & ep_words) / (len(q_words | ep_words) + 1e-9)
            scored.append((j, ep))
        scored.sort(key=lambda x: x[0], reverse=True)
        return [ep for score, ep in scored[:k] if score > 0]

    def format_few_shots(self, query: str) -> str:
        """Format retrieved episodes as few-shot text for the coordinator prompt."""
        similar = self.retrieve_similar(query)
        if not similar:
            return ""
        lines = ["\nSimilar past queries (for reference):"]
        for ep in similar:
            lines.append(
                f'  - "{ep["query"]}" -> intent={ep["intent"]}, '
                f'brands={ep["brands"]}, aspects={ep["aspects"]}'
            )
        return "\n".join(lines)


# Initialize episodic memory (loads from disk if exists)
episodic_memory = EpisodicMemory(CONFIG["episodic_file"])
print(f"Episodic memory: {len(episodic_memory.episodes)} stored episodes")


def compress_review(review: str) -> str:
    """Context compression: extract the most sentiment-dense sentences.

    """
    if len(review.split()) <= CONFIG["compress_min_words"]:
        return review

    blob = TextBlob(review)
    sentences = blob.sentences
    if len(sentences) <= CONFIG["compress_sentences"]:
        return review

    ranked = sorted(sentences, key=lambda s: abs(s.sentiment.polarity), reverse=True)
    return " ".join(str(s) for s in ranked[: CONFIG["compress_sentences"]])


# ── working memory across turns ──────────────────────────
# Module-level dicts persist for the lifetime of the Python kernel so that consecutive Gradio calls share brand reviews and conversation context.
# This is distinct from episodic_memory (disk-persisted, inter-session); the session cache lives only in RAM for the current kernel session.

_session_reviews: Dict[str, pd.DataFrame] = {}    # brand -> DataFrame (module-level global, NOT in AgentState)
_conversation_history: List[Dict] = []             # [{query, intent, brands, summary}, ...]


def clear_session():
    """Wipe all in-session data (call via Gradio "Clear Session" button)."""
    global _session_reviews, _conversation_history
    _session_reviews.clear()
    _conversation_history.clear()
    print("Session cleared.")


print("Session cache initialised (empty).")


Episodic memory: 0 stored episodes
Session cache initialised (empty).


In [9]:
# Cell H: Enhanced Search Agent (3-stage Hybrid RAG + Re-ranking)

class EnhancedSearchAgent:
    """Three-stage retrieval pipeline following the W7 Final MVP architecture:

    Stage 1 — Candidate generation:
        BM25 (sparse, keyword) + Bi-encoder (dense, semantic) scores fused
        with Reciprocal Rank Fusion (RRF).

    Stage 2 — Metadata filtering:
        Hard filter by brand name and optional rating range before embedding
        search, reducing the search space (W7: metadata filter).

    Stage 3 — Cross-encoder re-ranking:
        A computationally expensive but accurate cross-encoder scores
        (query, document) pairs for the top-K candidates (W7: reranker).
    """

    ASPECT_KW = {
        "battery":      ["battery", "charge", "power", "endurance", "drain"],
        "camera":       ["camera", "photo", "picture", "lens", "zoom", "image"],
        "screen":       ["screen", "display", "resolution", "brightness"],
        "performance":  ["speed", "lag", "processor", "fast", "smooth", "slow"],
        "price":        ["price", "cost", "cheap", "expensive", "value", "money"],
        "build_quality": ["build", "quality", "durable", "material", "solid", "feel"],
    }

    def __init__(self, df: pd.DataFrame):
        self.df = df.reset_index(drop=True)
        n = len(self.df)

        print(f"Building search index for {n:,} reviews...")

        # BM25 sparse index
        tokenized = [doc.lower().split() for doc in self.df["Reviews_Clean"]]
        self.bm25 = BM25Okapi(tokenized)
        print("  BM25 index: done")

        # Bi-encoder dense embeddings (pre-computed once)
        self.encoder = SentenceTransformer("all-MiniLM-L6-v2")
        self.doc_emb = self.encoder.encode(
            self.df["Reviews_Clean"].tolist(),
            batch_size=256, show_progress_bar=True, normalize_embeddings=True,
        )
        print(f"  Dense embeddings: {self.doc_emb.shape}")

        # Cross-encoder for re-ranking
        self.cross_enc = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
        print("  Cross-encoder: loaded\nSearch agent ready.")

    # ── internal helpers ──────────────────────────────────────────────────────

    def _bm25_scores(self, query: str, mask: np.ndarray = None) -> np.ndarray:
        scores = np.array(self.bm25.get_scores(query.lower().split()))
        if mask is not None:
            scores[~mask] = -1.0
        return scores

    def _dense_scores(self, query: str, mask: np.ndarray = None) -> np.ndarray:
        q_emb = self.encoder.encode([query], normalize_embeddings=True)
        sims = cosine_similarity(q_emb, self.doc_emb)[0]
        if mask is not None:
            sims[~mask] = -1.0
        return sims

    def _rrf(self, *rank_arrays, k: int = 60) -> np.ndarray:
        """Reciprocal Rank Fusion across arbitrarily many ranked lists."""
        n = len(rank_arrays[0])
        scores = np.zeros(n)
        for ranks in rank_arrays:
            for position, doc_idx in enumerate(ranks):
                scores[doc_idx] += 1.0 / (k + position + 1)
        return scores

    def _brand_mask(self, brand: str) -> np.ndarray:
        return self.df["Brand_Clean"].str.lower().str.contains(
            brand.lower(), na=False, regex=False
        ).values

    def _rerank(self, query: str, candidates: pd.DataFrame) -> pd.DataFrame:
        """Stage 3: cross-encoder re-ranking of candidate reviews."""
        pairs = [(query, r) for r in candidates["Reviews_Clean"]]
        scores = self.cross_enc.predict(pairs)
        candidates = candidates.copy()
        candidates["_score"] = scores
        return candidates.sort_values("_score", ascending=False).drop(columns=["_score"])

    # ── public API ────────────────────────────────────────────────────────────

    def search_for_brand(self, brand: str, aspect: str = "") -> pd.DataFrame:
        """Retrieve reviews for a specific brand, optionally filtered by aspect.

        Used for compare / analyze intents.
        """
        # Stage 2: metadata pre-filter to this brand only
        mask = self._brand_mask(brand)
        if mask.sum() == 0:
            available = self.df["Brand_Clean"].dropna().unique()
            similar = [b for b in available if brand.lower() in b.lower() or b.lower() in brand.lower()]
            msg = f"  No reviews found for brand: {brand}"
            if similar:
                msg += f" — similar brands in dataset: {similar[:3]}"
            print(msg)
            return pd.DataFrame(columns=["Reviews_Clean", "Brand_Clean", "Rating"])

        # Build aspect-augmented query for richer retrieval signal
        query = brand
        if aspect and aspect in self.ASPECT_KW:
            query += " " + " ".join(self.ASPECT_KW[aspect])

        # Stage 1: RRF over BM25 + dense
        bm25_s = self._bm25_scores(query, mask)
        dense_s = self._dense_scores(query, mask)
        rrf_s = self._rrf(np.argsort(-bm25_s), np.argsort(-dense_s))

        top_idx = np.argsort(-rrf_s)[: CONFIG["rerank_candidates"]]
        top_idx = [i for i in top_idx if mask[i]]

        if not top_idx:
            return self.df[mask].head(CONFIG["final_top_k"]).reset_index(drop=True)

        # Stage 3: cross-encoder re-ranking
        candidates = self.df.iloc[top_idx].copy()
        reranked = self._rerank(query, candidates)
        return reranked.head(CONFIG["final_top_k"]).reset_index(drop=True)

    def search_by_aspect(self, aspect: str, max_brands: int = 5) -> Dict[str, pd.DataFrame]:
        # Returns DataFrames — callers convert to List[Dict] before writing to AgentState
        """Retrieve reviews mentioning an aspect across all brands.

        Used for search intent (no brand constraint).
        """
        query = aspect
        if aspect in self.ASPECT_KW:
            query += " " + " ".join(self.ASPECT_KW[aspect])

        # Stage 1: full corpus RRF (no brand mask)
        bm25_s = self._bm25_scores(query)
        dense_s = self._dense_scores(query)
        rrf_s = self._rrf(np.argsort(-bm25_s), np.argsort(-dense_s))

        pool_size = min(CONFIG["rerank_candidates"] * max_brands, len(self.df))
        top_idx = np.argsort(-rrf_s)[:pool_size]
        candidates = self.df.iloc[top_idx].copy()

        # Stage 3: re-rank the full pool
        reranked = self._rerank(query, candidates)

        # Group by brand, pick top brands by review count
        results: Dict[str, pd.DataFrame] = {}
        for brand, grp in reranked.groupby("Brand_Clean"):
            results[brand] = grp.head(CONFIG["final_top_k"]).reset_index(drop=True)
            if len(results) >= max_brands:
                break

        return results


# Build the index (runs once; ~2-3 min on Colab CPU, ~30 s on GPU)
print("Building Enhanced Search Agent (first run takes a few minutes)...")
search_agent_obj = EnhancedSearchAgent(df_clean)
print("Search agent ready.")

Building Enhanced Search Agent (first run takes a few minutes)...
Building search index for 105,160 reviews...
  BM25 index: done


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/411 [00:00<?, ?it/s]

  Dense embeddings: (105160, 384)


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  Cross-encoder: loaded
Search agent ready.
Search agent ready.


In [10]:
# Cell I:  Aspect-level sentiment analysis + parallel execution

# In-session sentiment cache keyed by (review_hash, critique_hint)
# Using hint in the key ensures retry calls with different hints are never cached.
_sentiment_cache: Dict[int, Dict] = {}


def baseline_sentiment(text: str) -> Dict:
    """TextBlob baseline: fast, no API call, used as comparison model."""
    p = TextBlob(text).sentiment.polarity
    return {
        "sentiment": "positive" if p > CONFIG["pos_thresh"] else (
            "negative" if p < CONFIG["neg_thresh"] else "neutral"
        ),
        "polarity": p,
        "confidence": min(abs(p) + 0.5, 1.0),
        "aspects": [],
        "method": "baseline",
    }


def analyze_single_review(review: str, llm: ChatOpenAI, critique_hint: str = "") -> Dict:
    """Classify one review with CoT reasoning + aspect breakdown.

    critique_hint: non-empty on reflection retry runs.
    Injected into the prompt so the model knows what the evaluator found
    lacking — this is the core of the Evaluator-Optimizer pattern.

    Cache key includes critique_hint, so a retry with new guidance is
    always a fresh LLM call (no stale cache hits on retry passes).

    Context compression applied before the LLM call.
    Falls back to TextBlob on total failure.
    """
    key = hash((review[:100], critique_hint))
    if CONFIG["cache_results"] and key in _sentiment_cache:
        return _sentiment_cache[key]

    compressed = compress_review(review)

    # Build the hint block that gets substituted into {critique_hint}
    if critique_hint:
        hint_block = (
            "\nNote from previous analysis round: "
            f"{critique_hint}\n"
            "Please pay extra attention to those dimensions.\n"
        )
    else:
        hint_block = ""

    prompt = Prompts.SENTIMENT.format(review=compressed, critique_hint=hint_block)

    for attempt in range(CONFIG["llm_max_retries"]):
        try:
            out = llm.with_structured_output(DetailedSentimentResult).invoke([
                SystemMessage("You are a product review sentiment analyst. Think step by step."),
                HumanMessage(prompt),
            ])
            result = {
                "sentiment": out.overall_sentiment,
                "polarity": out.overall_polarity,
                "confidence": out.confidence,
                "reasoning": out.reasoning,
                "aspects": [
                    {"aspect": a.aspect, "sentiment": a.sentiment, "evidence": a.evidence}
                    for a in out.aspects
                ],
                "method": "gpt-4o-mini",
            }
            if CONFIG["cache_results"]:
                _sentiment_cache[key] = result
            return result
        except Exception as _e:
            if attempt == 0:
                print(f"  [Sentiment LLM error] {type(_e).__name__}: {_e}")
            if attempt < CONFIG["llm_max_retries"] - 1:
                time.sleep(0.5 * (2 ** attempt))

    print(f"  [Fallback] All LLM attempts failed for review snippet: {review[:60]!r}")
    fb = baseline_sentiment(review)
    fb["method"] = "baseline-fallback"
    return fb


def aggregate_results(brand: str, per_review: List[Dict], avg_rating: float) -> Dict:
    """Aggregate per-review results into brand-level metrics + aspect summary."""
    n = len(per_review)
    pos = sum(1 for r in per_review if r["sentiment"] == "positive")
    neg = sum(1 for r in per_review if r["sentiment"] == "negative")
    neu = n - pos - neg

    aspect_buckets: Dict[str, List[str]] = defaultdict(list)
    for r in per_review:
        for a in r.get("aspects", []):
            aspect_buckets[a["aspect"]].append(a["sentiment"])

    aspect_summary = {
        asp: {
            "positive": sentiments.count("positive") / len(sentiments),
            "negative": sentiments.count("negative") / len(sentiments),
            "neutral":  sentiments.count("neutral")  / len(sentiments),
            "mentions": len(sentiments),
        }
        for asp, sentiments in aspect_buckets.items()
    }

    return {
        "brand": brand,
        "method": per_review[0].get("method", "unknown") if per_review else "unknown",
        "total": n,
        "positive_count": pos,
        "negative_count": neg,
        "neutral_count":  neu,
        "positive_pct":   pos / n * 100 if n else 0,
        "negative_pct":   neg / n * 100 if n else 0,
        "avg_polarity":   float(np.mean([r["polarity"]             for r in per_review])) if per_review else 0,
        "avg_confidence": float(np.mean([r.get("confidence", 0.5)  for r in per_review])),
        "avg_rating":     avg_rating,
        "aspect_summary": aspect_summary,
    }


async def analyze_brand_async(
    brand: str,
    reviews_df: pd.DataFrame,
    llm: ChatOpenAI,
    critique_hint: str = "",
    max_n: int = None,
) -> List[Dict]:
    """Async analysis for one brand — runs LLM calls in a thread pool.

    Args:
        critique_hint: Empty on first pass. On retry, contains the reflection
                       agent's per-brand feedback injected into each review prompt.
        max_n: Review cap. Increased by analysis_agent for flagged brands on retry.

    W9 parallelization: brands run concurrently via asyncio.gather.
    """
    loop = asyncio.get_event_loop()
    cap = max_n if max_n is not None else CONFIG["max_reviews"]
    reviews = reviews_df["Reviews_Clean"].tolist()[:cap]
    avg_rating = float(reviews_df["Rating"].mean()) if "Rating" in reviews_df else 0.0

    llm_results = list(await asyncio.gather(
        *[loop.run_in_executor(None, analyze_single_review, rev, llm, critique_hint)
          for rev in reviews]
    ))
    baseline_results = [baseline_sentiment(rev) for rev in reviews]

    return [
        aggregate_results(brand, llm_results, avg_rating),
        {**aggregate_results(brand, baseline_results, avg_rating), "method": "baseline"},
    ]


def analysis_agent(state: AgentState) -> AgentState:
    """Run aspect-level CoT sentiment analysis across all brands in parallel.

    First pass : temperature=0 (deterministic baseline).
    Retry pass : temperature=0.4 (stochastic diversity) so re-running is not
                 equivalent to the first pass.

    On retry, three things change per brand (driven by reflection output):
    1. critique_hint injected into every review prompt
       — tells the model what the evaluator flagged as weak.
    2. max_n increased by 50% for brands flagged with increase_sample=True
       — more evidence to improve coverage and confidence.
    3. Cache key includes the hint, so retry calls always hit the LLM
       even if the same review was cached from the first pass.
    """
    print(f"\n{'='*60}\nANALYSIS AGENT\n{'='*60}")

    product_reviews = state.get("product_reviews", {})
    if not product_reviews:
        state["sentiment_results"] = []
        state["aspect_summary"] = {}
        return state

    is_retry = state.get("reflection_iter", 0) > 0

    # higher temperature on retry for output diversity
    temp = 0.4 if is_retry else 0.0
    llm = ChatOpenAI(model=CONFIG["model"], temperature=temp)
    if is_retry:
        print(f"  Retry run — temperature raised to {temp} for stochastic diversity")

    # Improvement 2: per-brand critique hints from reflection agent
    brand_hints: Dict[str, str] = state.get("reflection_brand_issues", {})

    # Improvement 3: brands flagged for a larger review sample
    increase_brands: List[str] = state.get("reflection_increase_sample", [])
    retry_max = int(CONFIG["max_reviews"] * 1.5)

    t0 = time.time()
    n_brands = len(product_reviews)
    print(f"Analyzing {n_brands} brand(s) in parallel (retry={is_retry}, temp={temp})...")

    async def _make_not_found(brand: str, reason: str):
        """Return a placeholder result for brands with no reviews."""
        return [{
            "brand": brand, "method": "not_found", "total": 0,
            "positive_count": 0, "negative_count": 0, "neutral_count": 0,
            "positive_pct": 0.0, "negative_pct": 0.0, "avg_polarity": 0.0,
            "avg_confidence": 0.0, "avg_rating": 0.0, "aspect_summary": {},
            "error": reason,
        }]

    async def run_all():
        tasks = []
        for brand, records in product_reviews.items():
            # Convert List[Dict] back to DataFrame locally — DataFrames never travel in state
            if not records:
                not_found_note = state.get("_search_notes", {}).get(brand, "No reviews found in dataset.")
                print(f"  [Warning] Skipping {brand}: empty record list. ({not_found_note})")
                tasks.append(_make_not_found(brand, not_found_note))
                continue
            df = pd.DataFrame(records)
            if "Reviews_Clean" not in df.columns:
                print(f"  [Warning] Skipping {brand}: Reviews_Clean column missing.")
                continue

            hint  = brand_hints.get(brand, "")
            max_n = retry_max if (is_retry and brand in increase_brands) else CONFIG["max_reviews"]

            tasks.append(analyze_brand_async(brand, df, llm, critique_hint=hint, max_n=max_n))

        return await asyncio.gather(*tasks) if tasks else [] # Remember to add 'if tasks else []' here

    brand_batches = asyncio.run(run_all())
    elapsed = time.time() - t0
    print(f"Done in {elapsed:.1f}s")

    all_results = [row for batch in brand_batches for row in batch]

    llm_results = [r for r in all_results if r["method"] not in ("baseline", "baseline-fallback")]
    aspect_summary = {r["brand"]: r["aspect_summary"] for r in llm_results}

    # comparison_table stored as List[Dict] — JSON-serializable (no pd.DataFrame in state)
    comp = [{
        "Brand":          r["brand"],
        "Reviews":        r["total"],
        "Avg Rating":     round(r["avg_rating"], 2),
        "Positive %":     round(r["positive_pct"], 1),
        "Negative %":     round(r["negative_pct"], 1),
        "Avg Polarity":   round(r["avg_polarity"], 3),
        "Avg Confidence": round(r["avg_confidence"], 3),
    } for r in llm_results] if llm_results else None

    state["sentiment_results"] = all_results
    state["aspect_summary"]    = aspect_summary
    state["comparison_table"]  = comp

    for r in llm_results:
        print(f"  {r['brand']}: {r['positive_pct']:.0f}% positive | "
              f"conf={r['avg_confidence']:.2f} | aspects={list(r['aspect_summary'].keys())}")

    return state

In [11]:
# Cell J: Reflection Agent (Evaluator-Optimizer pattern)

def reflection_agent(state: AgentState) -> AgentState:
    """Quality gate between analysis and report.

    W9 Evaluator-Optimizer — three outputs drive the retry:
    1. reflection_brand_issues  : per-brand critique string injected into
                                  the next sentiment prompt (improve focus).
    2. reflection_increase_sample: brands where pulling more reviews helps
                                  (analysis_agent raises review cap by 50%).
    3. reflection_score         : overall score; below threshold -> retry.

    This means each retry pass genuinely changes what the analysis agent
    does — different temperature, different prompt guidance, different data
    volume — rather than blindly repeating the same call.
    """
    iter_n = state.get("reflection_iter", 0)
    state["reflection_iter"] = iter_n + 1   # must increment in node, not in routing fn
    print(f"\n{'='*60}\nREFLECTION AGENT  (iteration {iter_n + 1})\n{'='*60}")

    results    = state.get("sentiment_results", [])
    llm_results = [r for r in results if r.get("method") not in ("baseline", "baseline-fallback")]

    if not llm_results:
        state["reflection_score"]           = 0
        state["reflection_issues"]          = ["No LLM results to evaluate"]
        state["reflection_brand_issues"]    = {}
        state["reflection_increase_sample"] = []
        return state

    summary = "\n".join(
        f"- {r['brand']}: {r['total']} reviews, {r['positive_pct']:.0f}% positive, "
        f"avg_confidence={r['avg_confidence']:.2f}, aspects={list(r['aspect_summary'].keys())}"
        for r in llm_results
    )

    llm    = ChatOpenAI(model=CONFIG["model"], temperature=0)
    prompt = Prompts.REFLECTION.format(
        results_summary=summary,
        threshold=CONFIG["reflection_threshold"],
    )

    try:
        out = llm.with_structured_output(ReflectionScore).invoke([
            SystemMessage("You are a rigorous quality reviewer. Be objective."),
            HumanMessage(prompt),
        ])
        score, issues, brand_critiques = out.score, out.issues, out.brand_critiques
    except Exception as e:
        print(f"  Reflection LLM error: {e} — defaulting to pass.")
        score, issues, brand_critiques = 8, [], []

    # Unpack per-brand critiques into state fields consumed by analysis_agent
    brand_hints: Dict[str, str] = {}
    increase_sample: List[str]  = []

    for bc in brand_critiques:
        if bc.issues:
            brand_hints[bc.brand] = "; ".join(bc.issues)
        if bc.increase_sample:
            increase_sample.append(bc.brand)

    state["reflection_score"]           = score
    state["reflection_issues"]          = issues
    state["reflection_brand_issues"]    = brand_hints
    state["reflection_increase_sample"] = increase_sample

    print(f"  Score: {score}/10")
    if issues:
        print(f"  Overall: {'; '.join(issues)}")
    for brand, hint in brand_hints.items():
        extra = " [will increase sample]" if brand in increase_sample else ""
        print(f"  {brand}{extra}: {hint}")

    return state


def _route_after_reflection(state: AgentState) -> str:
    """Retry analysis if quality is below threshold, else proceed to visualization.
    NOTE: reflection_iter is already incremented by reflection_agent (the node).
    Routing functions must never mutate LangGraph state.
    """
    score    = state.get("reflection_score", 10)
    itr      = state.get("reflection_iter", 0)   # already incremented by reflection_agent
    max_iter = CONFIG.get("reflection_max_iter", 2)

    if score < CONFIG["reflection_threshold"] and itr <= max_iter:
        print(f"  -> Retrying analysis (score={score} < {CONFIG['reflection_threshold']}, "
              f"attempt {itr}/{max_iter}).")
        return "retry"

    print(f"  -> Proceeding to report (score={score}).")
    return "proceed"

In [12]:
# Cell K: Tool definitions using LangChain @tool decorator

# Shared state written by tools, read back into AgentState after the ReAct loop
_tool_state: Dict = {"search_results": {}, "brands_queried": []}


@tool
def search_brand_reviews(brand_name: str, aspect: str = "") -> str:
    """Search for mobile phone product reviews for a specific brand.

    Call this before analyze_sentiment. Optionally narrow results to an aspect.

    Args:
        brand_name: Exact brand name to search for (e.g. "Samsung", "Apple", "Nokia").
        aspect: Product feature to focus on. One of: battery, camera, screen,
                performance, price, build_quality. Leave empty for all reviews.

    Returns:
        JSON string: {brand, reviews_found, sample_reviews (first 3)}.
    """
    df = search_agent_obj.search_for_brand(brand_name, aspect)
    _tool_state["search_results"][brand_name] = df
    _tool_state["brands_queried"].append(brand_name)
    result = {
        "brand": brand_name,
        "reviews_found": len(df),
        "sample_reviews": df["Reviews_Clean"].head(3).tolist() if len(df) else [],
    }
    if len(df) == 0:
        available = search_agent_obj.df["Brand_Clean"].dropna().unique().tolist()
        similar = [b for b in available if brand_name.lower() in b.lower() or b.lower() in brand_name.lower()]
        result["note"] = (
            f"No reviews found for '{brand_name}'" +
            (f". Closest matches in dataset: {similar[:3]}" if similar else ". Brand not found in dataset.")
        )
    return json.dumps(result)


@tool
def find_phones_by_feature(feature: str, max_brands: int = 5) -> str:
    """Search across all brands for reviews mentioning a specific product feature.

    Use for open-ended queries like 'find phones with good camera'.
    Do NOT use when a specific brand is already known — use search_brand_reviews instead.

    Args:
        feature: Product feature. One of: battery, camera, screen,
                 performance, price, build_quality.
        max_brands: Maximum number of top brands to return (default 5).

    Returns:
        JSON string: {feature, brands_found, review_counts per brand}.
    """
    results = search_agent_obj.search_by_aspect(feature, max_brands)
    for brand, df in results.items():
        _tool_state["search_results"][brand] = df
        _tool_state["brands_queried"].append(brand)
    return json.dumps({
        "feature": feature,
        "brands_found": list(results.keys()),
        "review_counts": {b: len(df) for b, df in results.items()},
    })


# ── demonstration: 5-step tool calling loop ──────────────────────────────

def demo_w8_tool_calling():
    """Demonstrate the W8 five-step tool-calling loop explicitly.

    Step 1  User message
    Step 2  LLM picks a tool and returns a structured call (JSON schema)
    Step 3  Application executes the tool (LLM never runs it)
    Step 4  Tool result returned to application
    Step 5  Application feeds result back to LLM → final answer
    """
    print("=== W8: Five-step tool calling loop ===")
    llm = ChatOpenAI(model=CONFIG["model"], temperature=0)
    model_with_tools = llm.bind_tools([search_brand_reviews, find_phones_by_feature])

    # Step 1
    user_query = "How many Samsung reviews are available?"
    messages = [HumanMessage(user_query)]
    print(f"Step 1 — User: {user_query}")

    # Step 2: LLM decides which tool and what args
    response = model_with_tools.invoke(messages)
    print(f"Step 2 — LLM tool call: {response.tool_calls}")

    if response.tool_calls:
        tc = response.tool_calls[0]
        # Step 3+4: application runs the tool (LLM is not involved)
        tool_result = search_brand_reviews.invoke(tc["args"])
        print(f"Step 3+4 — Tool result: {tool_result[:200]}")

        # Step 5: feed result back, get final natural-language answer
        messages += [response, ToolMessage(tool_result, tool_call_id=tc["id"])]
        final = model_with_tools.invoke(messages)
        print(f"Step 5 — Final answer: {final.content}")

print("Tool definitions ready. Run demo_w8_tool_calling() to see the W8 loop.")

Tool definitions ready. Run demo_w8_tool_calling() to see the W8 loop.


In [13]:
# Cell L:  Coordinator (structured output + episodic memory + session cache)
#          + Cache Inject Agent
#          + ReAct Search Agent (create_react_agent)

def coordinator_agent(state: AgentState) -> AgentState:
    """Parse query intent with structured output.

    W10: Injects (1) episodic memory few-shots and (2) session-cache context into
    the coordinator prompt so the LLM can decide which brands need a fresh search
    vs which can be served from the in-memory cache (reuse_brands vs new_brands).
    """
    print(f"\n{'='*60}\nCOORDINATOR\n{'='*60}")
    print(f"Query: {state['query']}")

    llm = ChatOpenAI(model=CONFIG["model"], temperature=0)

    # episodic memory: retrieve similar past queries as few-shots
    few_shots = episodic_memory.format_few_shots(state["query"])

    # working memory: build conversation context from session history
    hist = state.get("conversation_history", [])
    if hist:
        lines = ["\nConversation so far:"]
        for turn in hist[-3:]:   # keep last 3 turns to avoid prompt bloat
            lines.append(
                f'  Turn {turn["turn"]}: "{turn["query"]}" '
                f'-> intent={turn["intent"]}, brands={turn["brands"]}'
            )
            if turn.get("summary"):
                lines.append(f'    Summary: {turn["summary"]}')
        conversation_ctx = "\n".join(lines)
    else:
        conversation_ctx = ""

    # Tell coordinator which brands are already in the session cache
    cached = list(_session_reviews.keys())
    cached_brands = ", ".join(cached) if cached else "(none)"

    prompt = Prompts.COORDINATOR.format(
        query=state["query"],
        few_shots=few_shots,
        conversation_ctx=conversation_ctx,
        cached_brands=cached_brands,
    )

    try:
        result = llm.with_structured_output(QueryIntent).invoke([
            SystemMessage(
                "You are a product review query parser. Be precise.\n"
                "If a brand's reviews are already cached, put it in reuse_brands "
                "and do NOT include it in new_brands."
            ),
            HumanMessage(prompt),
        ])
        state["intent"]       = result.intent
        state["target_brands"] = result.brands
        state["aspects"]      = result.aspects
        state["reuse_brands"] = result.reuse_brands
        state["new_brands"]   = result.new_brands
        state["error"]        = None

        if result.intent == "off_topic":
            state["guidance_message"] = (
                "This system is designed to analyse Amazon mobile phone reviews. "
                "Try queries like:\n"
                "\u2022 \"Compare Samsung and Apple on battery life\"\n"
                "\u2022 \"Analyse Nokia reviews\"\n"
                "\u2022 \"Find phones with good camera\"\n"
                f"\n(Coordinator reasoning: {result.reasoning})"
            )
            state["new_brands"] = []
            state["reuse_brands"] = []
            print("  Off-topic query detected. Guidance set.")
        else:
            state["guidance_message"] = None


        print(f"  Intent: {result.intent}  |  Brands: {result.brands}")
        print(f"  Reuse (cached): {result.reuse_brands}  |  New (search): {result.new_brands}")
        if few_shots:
            n_shots = len(episodic_memory.retrieve_similar(state["query"]))
            print(f"  Episodic memory: {n_shots} similar examples injected")
        if cached:
            print(f"  Session cache holds: {cached}")
    except Exception as e:
        state["error"] = str(e)
        state["reuse_brands"] = []
        state["new_brands"]   = state.get("target_brands", [])
        print(f"  Coordinator error: {e}")

    return state


def cache_inject_agent(state: AgentState) -> AgentState:
    """Working memory: inject cached brand reviews into product_reviews.

    Runs before search so that the search agent only needs to handle new_brands.
    If all brands are already cached, search is skipped entirely via the
    _route_after_cache conditional edge.

    Serialization note: _session_reviews stores pd.DataFrames (module-level global,
    never serialized). product_reviews in AgentState stores List[Dict] records only.
    """
    print(f"\n{'='*60}\nCACHE INJECT AGENT\n{'='*60}")
    product_reviews = dict(state.get("product_reviews", {}))

    for brand in state.get("reuse_brands", []):
        if brand in _session_reviews:
            # Convert DataFrame -> List[Dict] before writing to state
            product_reviews[brand] = _session_reviews[brand].to_dict("records")
            print(f"  Injected {len(_session_reviews[brand])} cached reviews for: {brand} (from session cache)")
        else:
            # Brand was listed as reuse but not actually in cache — promote to new
            print(f"  WARNING: {brand} listed as reuse but not in cache — will search")
            nb_list = state.get("new_brands", [])
            if brand not in nb_list:
                state["new_brands"] = nb_list + [brand]

    state["product_reviews"] = product_reviews
    return state


def _route_after_cache(state: AgentState) -> str:
    """If all brands are cached, skip search and go straight to analysis."""

    if state.get("new_brands") or state.get("intent") == "search":
        return "search"
    print("  All brands served from session cache — skipping search.")
    return "analysis"



def react_search_agent(state: AgentState) -> AgentState:
    """Dynamic search using a ReAct agent that calls tools iteratively.

    Uses LangGraph's create_react_agent, which implements the full
    Think -> Act -> Observe -> Think cycle. Only searches new_brands (brands
    not already in the session cache). After retrieval, results are saved to
    _session_reviews for future turns.

    Tool results are written to _tool_state and transferred to AgentState.
    """
    print(f"\n{'='*60}\nREACT SEARCH AGENT\n{'='*60}")

    brands_to_search = state.get("new_brands", state.get("target_brands", []))
    if not brands_to_search and state.get("intent") != "search":
        print("  No new brands to search.")
        return state


    # Reset shared tool state for this query
    _tool_state["search_results"] = {}
    _tool_state["brands_queried"] = []

    llm = ChatOpenAI(model=CONFIG["model"], temperature=0)
    tools = [search_brand_reviews, find_phones_by_feature]

    system = (
        f"You are a product review search agent. Use the tools to find reviews.\n\n"
        f"Query intent: {state['intent']}\n"
        f"Brands to search (NEW — not cached): {brands_to_search}\n"
        f"Aspects to focus on: {state['aspects']}\n\n"
        "Strategy:\n"
        "- compare/analyze intent: call search_brand_reviews for each brand listed above.\n"
        "  If aspects are given, pass the first aspect to filter results.\n"
        "- search intent: call find_phones_by_feature for each aspect listed above.\n"
        "Stop when all listed brands have been searched. Do NOT analyze sentiment."
    )

    agent = create_react_agent(llm, tools, prompt=system)

    try:
        agent.invoke(
            {"messages": [HumanMessage(f"Search reviews for: {state['query']}")]},
            config={"recursion_limit": 50},
        )
    except Exception as e:
        from langgraph.errors import GraphRecursionError
        if isinstance(e, GraphRecursionError):
            print(f"  ReAct hit step limit — merging partial results collected so far.")
        else:
            print(f"  ReAct search error: {e}")
            state["error"] = str(e)

    # Always merge results regardless of whether agent finished cleanly or hit a limit.
    # _tool_state holds whatever tools managed to collect before any exception.
    product_reviews = dict(state.get("product_reviews", {}))
    new_results_raw = dict(_tool_state["search_results"])  # brand -> pd.DataFrame

    # Fallback: directly fetch any brands the ReAct agent never got to
    missed = [b for b in brands_to_search if b not in new_results_raw]
    if missed:
        print(f"  ReAct missed {missed} — fetching directly")
        for brand in missed:
            df_direct = search_agent_obj.search_for_brand(brand)
            new_results_raw[brand] = df_direct

    if new_results_raw:
        new_results = {brand: df_r.to_dict("records") for brand, df_r in new_results_raw.items()}
        product_reviews.update(new_results)
        state["product_reviews"] = product_reviews

        for brand, df_records in new_results_raw.items():
            _session_reviews[brand] = df_records
            print(f"  Cached {len(df_records)} reviews for: {brand}")

        total = sum(len(df) for df in state["product_reviews"].values())
        print(f"  Total product_reviews: {len(state['product_reviews'])} brand(s), {total} reviews")
    else:
        if not state.get("product_reviews"):
            state["product_reviews"] = {}

    return state


In [14]:
# Cell M: Visualization Agent
import plotly.express as px
import plotly.graph_objects as go
def _generate_ai_commentary(state: dict) -> dict:
    """Use LLM to generate professional commentary for each chart."""
    print("  Generating AI chart commentary...")
    llm = ChatOpenAI(model=CONFIG["model"], temperature=0.2)

    summary_data = []
    for r in state.get("sentiment_results", []):
        if r.get("method") not in ("baseline", "baseline-fallback"):
            summary_data.append(f"{r['brand']}: {r['positive_pct']:.0f}% Positive, Total Reviews: {r['total']}. Aspects: {list(r['aspect_summary'].keys())}")

    if not summary_data:
        return {}
    prompt = f"""You are a senior management consultant. Based on this product review data, write ONE sharp, professional sentence of analytical commentary for each of these chart types.
    Do not mention the chart name directly, just provide the insight the chart reveals.

    Data Context:
    {chr(10).join(summary_data)}
    """

    try:
        out = llm.with_structured_output(ChartCommentary).invoke([
            SystemMessage(content="You are a data analyst producing a BI report. Keep it concise and insightful."),
            HumanMessage(content=prompt)
        ])
        return out.dict()
    except Exception as e:
        print(f"  Commentary generation failed: {e}")
        return {}



def visualization_agent(state: dict) -> dict:
    """Generate high-quality, aesthetically pleasing Plotly charts in MBB style."""
    print(f"\n{'='*60}\nVISUALIZATION AGENT (Pro Consulting Edition)\n{'='*60}")

    charts: Dict = {}
    results = state.get("sentiment_results", [])
    llm_results = [r for r in results if r.get("method") not in ("baseline", "baseline-fallback")]

    if not llm_results:
        state["viz_charts"] = charts
        return state

    df_r = pd.DataFrame(llm_results)

    # --- 1. MBB Consulting Style Global Settings ---
    colors = {
        "pos": "#0D5257",  # Deep Teal (Strength/Advantage)
        "neu": "#9EA3A7",  # Slate Gray (Neutral)
        "neg": "#8B1E28",  # Burgundy (Risk/Weakness)
        "bg": "rgba(0,0,0,0)",
        "grid": "#E8EAEB",
        "text": "#212529"
    }

    consulting_layout = dict(
        font=dict(family="Arial, Helvetica, sans-serif", color=colors["text"], size=13),
        plot_bgcolor=colors["bg"],
        paper_bgcolor=colors["bg"],
        hoverlabel=dict(
            bgcolor="white", font_size=13,
            font_family="Arial, Helvetica, sans-serif",
            bordercolor="#212529"
        ),
        colorway=[colors["pos"], colors["neu"], colors["neg"]],
        title_font=dict(size=18, family="Arial, Helvetica, sans-serif", color="#000000")
    )

    clean_axes = dict(
        showgrid=True, gridcolor=colors["grid"],
        zeroline=False, showline=False,
        tickfont=dict(color="#64748b")
    )

    # ==========================================
    # Chart 1: Overall Brand Sentiment Distribution
    # ==========================================
    brands = df_r["brand"].unique()
    fig1 = make_subplots(
        rows=1, cols=len(brands),
        specs=[[{"type": "pie"}] * len(brands)],
        subplot_titles=list(brands)
    )

    for i, brand in enumerate(brands, 1):
        row = df_r[df_r["brand"] == brand].iloc[0]
        fig1.add_trace(
            go.Pie(
                labels=["Positive", "Neutral", "Negative"],
                values=[row["positive_count"], row["neutral_count"], row["negative_count"]],
                name=brand,
                hole=0.65, # Enlarge center hole
                textposition='outside',
                textinfo='label+percent',
                marker=dict(
                    colors=[colors["pos"], colors["neu"], colors["neg"]],
                    line=dict(color='#ffffff', width=3)
                ),
                textfont=dict(size=13, color=colors["text"]),
                direction='clockwise',
                sort=False # Lock slice order (Pos, Neu, Neg)
            ),
            row=1, col=i,
        )
    fig1.update_layout(
        **consulting_layout,
        title_text="<b>Overall Brand Sentiment Distribution</b>",
        showlegend=False,
        margin=dict(t=80, b=40, l=80, r=80)
    )
    charts["sentiment_distribution"] = fig1
    print("  Chart 1: Sentiment Donut updated")

    # ==========================================
    # Chart 2: Aspect Sentiment Treemap
    # ==========================================
    asp_summary = state.get("aspect_summary", {})
    if asp_summary and len(brands) > 0:
        treemap_data = []
        for brand, aspects in asp_summary.items():
            for asp, metrics in aspects.items():
                if metrics["mentions"] > 0:
                    treemap_data.append({
                        "Brand": brand,
                        "Aspect": asp,
                        "Mentions": metrics["mentions"],
                        "Positive %": metrics["positive"] * 100
                    })

        df_tree = pd.DataFrame(treemap_data)

        if not df_tree.empty:
            fig2 = px.treemap(
                df_tree,
                path=[px.Constant("All Brands"), "Brand", "Aspect"],
                values="Mentions",
                color="Positive %",
                color_continuous_scale=[colors["neg"], "#E8EAEB", colors["pos"]],
                color_continuous_midpoint=50,
                custom_data=["Mentions", "Positive %"]
            )

            fig2.update_traces(
                textinfo="label+percent parent",
                hovertemplate="<b>%{label}</b><br>Mentions: %{customdata[0]}<br>Positive: %{customdata[1]:.1f}%<extra></extra>",
                marker=dict(line=dict(color='#FFFFFF', width=1.5)),
            )

            fig2.update_layout(
                **consulting_layout,
                title_text="<b>Product Feature Landscape (Treemap)</b>",
                coloraxis_colorbar=dict(title="Positive %", thicknessmode="pixels", thickness=15)
            )
            charts["aspect_heatmap"] = fig2
            print("  Chart 2: Aspect Treemap added")

    # ==========================================
    # Chart 3: Focus Brand Aspect Comparison (Grouped Bar)
    # ==========================================
    if asp_summary:
        focus_brand = brands[0]
        aspects_data = asp_summary.get(focus_brand, {})

        if aspects_data:
            sorted_aspects = sorted(aspects_data.items(), key=lambda x: x[1]["mentions"], reverse=True)
            top_aspects = sorted_aspects[:15]

            asp_names = [a[0] for a in top_aspects]
            pos_vals = [a[1]["positive"] * 100 for a in top_aspects]
            neg_vals = [a[1]["negative"] * 100 for a in top_aspects]

            fig3 = go.Figure()
            # 正評條形
            fig3.add_trace(go.Bar(
                y=asp_names, x=pos_vals, name='Positive %',
                orientation='h', marker_color=colors["pos"],
                hovertemplate="%{x:.1f}% Positive"
            ))
            # 負評條形 (改為同方向)
            fig3.add_trace(go.Bar(
                y=asp_names, x=neg_vals, name='Negative %',
                orientation='h', marker_color=colors["neg"],
                hovertemplate="%{x:.1f}% Negative"
            ))

            fig3.update_layout(
                **consulting_layout,
                title_text=f"<b>{focus_brand}: Feature Sentiment Breakdown</b>",
                barmode='group',
                yaxis_autorange="reversed",
                legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
            )
            fig3.update_xaxes(title="Percentage (%)", **clean_axes)
            fig3.update_yaxes(**clean_axes)

            charts["diverging_bar"] = fig3
            print("  Chart 3: Grouped Bar added")

    # ==========================================
    # Chart 4: Volume vs. Satisfaction Matrix (Fixed Range)
    # ==========================================
    if len(df_r) > 0:

        professional_sequence = px.colors.qualitative.Prism
        unique_brands = df_r["brand"].unique()
        brand_color_map = {b: professional_sequence[i % len(professional_sequence)] for i, b in enumerate(unique_brands)}

        fig4 = px.scatter(
            df_r, x="avg_rating", y="positive_pct",
            size="total",
            color="brand",
            text="brand",
            color_discrete_map=brand_color_map,
            title="<b>Brand Competitiveness Matrix (Strategic Landscape)</b>",
            labels={"avg_rating": "Avg Rating", "positive_pct": "Positive %", "total": "Volume"},
            size_max=50
        )

        fig4.update_traces(
            mode='markers+text',
            textposition='top center',
            marker=dict(line=dict(color='#ffffff', width=1.5), opacity=0.8)
        )

        fig4.update_layout(
            **consulting_layout,
            xaxis=dict(
                range=[1, 5.5],
                tickvals=[1, 2, 3, 4, 5],
                title="Market Rating Score (1-5)"
            ),
            yaxis=dict(
                range=[-15, 115],
                title="Positive Sentiment (%)"
            ),
            margin=dict(t=100, b=50, l=60, r=60),
            legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1)
        )

        charts["brand_comparison"] = fig4
        print(f"  Chart 4 updated: Ensuring all {len(unique_brands)} brands are visible.")
    state["viz_charts"] = charts
    # Generate AI commentary so report_generator needs no LLM calls
    state["chart_commentary"] = _generate_ai_commentary(state)
    return state

In [15]:
# Cell N: Report Generator

from docx import Document
from docx.shared import Inches
import os
import json
from pydantic import BaseModel, Field

class ChartCommentary(BaseModel):
    sentiment_distribution: str = Field(description="Commentary on overall brand sentiment shares")
    aspect_heatmap: str = Field(description="Commentary on specific feature strengths/weaknesses")
    diverging_bar: str = Field(description="Commentary on the polarizing aspects for the main brand")
    brand_comparison: str = Field(description="Commentary on volume vs satisfaction matrix")

def _extract_insights(state: dict) -> dict:
    """Derive key findings, risks, and recommendations from results."""
    insights = {"key_findings": [], "risks": [], "recommendations": []}

    llm_results = [r for r in state.get("sentiment_results", [])
                   if r.get("method") not in ("baseline", "baseline-fallback")]
    if not llm_results:
        return insights

    df_r = pd.DataFrame(llm_results)
    top = df_r.loc[df_r["positive_pct"].idxmax()]
    bot = df_r.loc[df_r["positive_pct"].idxmin()]

    insights["key_findings"].append(
        f"Highest sentiment: {top['brand']} — {top['positive_pct']:.1f}% positive "
        f"(avg rating {top['avg_rating']:.2f}/5)"
    )
    if len(df_r) > 1:
        gap = top["positive_pct"] - bot["positive_pct"]
        insights["key_findings"].append(
            f"Sentiment gap: {gap:.1f}pp between {top['brand']} and {bot['brand']}"
        )

    score = state.get("reflection_score")
    if score is not None:
        insights["key_findings"].append(f"Analysis quality score: {score}/10")

    for brand, aspects in state.get("aspect_summary", {}).items():
        weak = [(a, d["positive"]) for a, d in aspects.items() if d["positive"] < 0.40]
        if weak:
            names = ", ".join(a for a, _ in weak)
            insights["risks"].append(f"{brand}: Low positive rate on {names}")

    insights["recommendations"].append(
        f"Study {top['brand']}'s strengths ({top['positive_pct']:.1f}% positive) "
        f"to close the gap with lower-performing brands."
    )
    return insights


def report_generator(state: dict) -> dict:
    """Produce a Word document with charts, comparison table, aspect breakdown, and insights."""
    print(f"\n{'='*60}\nREPORT GENERATOR (Enhanced with Aspect Details)\n{'='*60}")

    doc = Document()
    insights = _extract_insights(state)
    commentaries = state.get("chart_commentary", {})
    now = datetime.now().strftime("%Y-%m-%d %H:%M")

    # 1. Title & Header
    doc.add_heading("Product Review Intelligence Report", 0)
    doc.add_paragraph(f"Query: {state['query']}   |   Generated: {now}")

    # 2. Executive Summary
    doc.add_heading("Executive Summary", 1)
    if insights["key_findings"]:
        for f in insights["key_findings"]:
            doc.add_paragraph(f, style="List Bullet")

    if insights["risks"]:
        doc.add_heading("Risk Flags", 1)
        for r in insights["risks"]:
            doc.add_paragraph(r, style="List Bullet")

    # 3. Visual Insights (Charts & AI Commentary)
    charts = state.get("viz_charts", {})
    if charts:
        doc.add_heading("Visual Data Analysis", 1)
        chart_config = {
            "sentiment_distribution": "1. Overall Brand Sentiment Distribution",
            "aspect_heatmap": "2. Aspect Satisfaction Matrix",
            "diverging_bar": "3. Sentiment Polarization by Feature",
            "brand_comparison": "4. Brand Competitiveness Landscape"
        }

        for chart_key, title in chart_config.items():
            if chart_key in charts:
                fig = charts[chart_key]
                doc.add_heading(title, 2)
                img_path = f"tmp_{chart_key}.png"
                try:
                    fig.write_image(img_path, width=800, height=450, scale=1.5)

                    doc.add_picture(img_path, width=Inches(6.0))
                    if os.path.exists(img_path):
                        os.remove(img_path)
                except Exception as e:
                    doc.add_paragraph(f"[See interactive chart in dashboard]")

                if chart_key in commentaries:
                    p = doc.add_paragraph()
                    p.add_run("🤖 AI Insight: ").bold = True
                    p.add_run(commentaries[chart_key])

    # ==========================================
    # 4. NEW: Detailed Feature Analysis (Aspect Level)
    # ==========================================
    aspect_summary = state.get("aspect_summary", {})
    if aspect_summary:
        doc.add_heading("Detailed Feature Analysis", 1)

        for brand, aspects in aspect_summary.items():
            doc.add_heading(f"Brand Deep-Dive: {brand}", level=2)

            if aspects:
                # 建立特徵表格
                table = doc.add_table(rows=1, cols=4)
                table.style = "Table Grid"

                # 設置表頭
                hdr_cells = table.rows[0].cells
                hdr_cells[0].text = 'Aspect / Feature'
                hdr_cells[1].text = 'Pos %'
                hdr_cells[2].text = 'Neg %'
                hdr_cells[3].text = 'Mentions'

                # 依據提及次數排序，取前 12 名最重要的特徵
                sorted_aspects = sorted(aspects.items(), key=lambda x: x[1]['mentions'], reverse=True)

                for asp_name, metrics in sorted_aspects[:12]:
                    row_cells = table.add_row().cells
                    row_cells[0].text = str(asp_name)
                    row_cells[1].text = f"{metrics['positive']*100:.0f}%"
                    row_cells[2].text = f"{metrics['negative']*100:.0f}%"
                    row_cells[3].text = str(metrics['mentions'])
            else:
                doc.add_paragraph("No specific aspect data found for this brand.")

    # 5. Appendix (Original Data Tables)
    # comparison_table is List[Dict] in state — convert to DataFrame for report tables
    _comp_records = state.get("comparison_table")
    comp = pd.DataFrame(_comp_records) if _comp_records else None
    if comp is not None:
        doc.add_heading("Appendix: Brand Comparison Data", 1)
        t = doc.add_table(rows=1, cols=len(comp.columns))
        t.style = "Table Grid"
        for j, col in enumerate(comp.columns):
            t.rows[0].cells[j].text = col
        for _, row in comp.iterrows():
            cells = t.add_row().cells
            for j, val in enumerate(row):
                cells[j].text = str(val)

    # 6. Save Report
    path = f"report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.docx"
    doc.save(path)
    state["report_path"] = path
    print(f"  Saved: {path}")

    # Memory & History Updates
    if "intent" in state and "target_brands" in state:
        episodic_memory.store(
            state["query"], state.get("intent", ""),
            state.get("target_brands", []), state.get("aspects", []),
        )

    turn_summary = []
    for r in state.get("sentiment_results", []):
        if r.get("method") not in ("baseline", "baseline-fallback"):
            turn_summary.append(f"{r.get('brand', '?')}: {r.get('positive_pct', 0):.0f}% pos")

    _conversation_history.append({
        "turn": len(_conversation_history) + 1,
        "query": state.get("query", ""),
        "intent": state.get("intent", ""),
        "brands": state.get("target_brands", []),
        "summary": "; ".join(turn_summary) if turn_summary else "no results",
    })
    state["conversation_history"] = list(_conversation_history)

    return state

In [16]:
# Cell O: LangGraph workflow assembly

def build_workflow():

    wf = StateGraph(AgentState)

    wf.add_node("coordinator",   coordinator_agent)
    wf.add_node("cache_inject",  cache_inject_agent)
    wf.add_node("search",        react_search_agent)
    wf.add_node("analysis",      analysis_agent)
    wf.add_node("reflection",    reflection_agent)
    wf.add_node("visualization", visualization_agent)
    #wf.add_node("report",        report_generator)

    wf.set_entry_point("coordinator")
    wf.add_edge("coordinator", "cache_inject")

    # Conditional: skip search if all brands already cached
    wf.add_conditional_edges(
        "cache_inject",
        _route_after_cache,
        {"search": "search", "analysis": "analysis"},
    )

    wf.add_edge("search",        "analysis")
    wf.add_edge("analysis",      "reflection")

    wf.add_conditional_edges(
        "reflection",
        _route_after_reflection,
        {"retry": "analysis", "proceed": "visualization"},
    )

    # wf.add_edge("visualization", "report")
    # wf.add_edge("report", END)
    wf.add_edge("visualization", END)

    return wf.compile()


app = build_workflow()
print("LangGraph workflow compiled.")
print("  coordinator -> cache_inject -> [search (if new)] -> analysis -> reflection -> viz -> report")


def run_query(query: str) -> AgentState:
    """Run a natural language query through the full pipeline.

    On each call the coordinator checks _session_reviews (module-level) and
    injects cached brands so repeated or follow-up queries skip re-fetching.
    """
    initial = AgentState(
        query=query, intent="", target_brands=[], aspects=[],
        reuse_brands=[], new_brands=[],
        product_reviews={}, sentiment_results=[], aspect_summary={},
        comparison_table=None,
        reflection_score=None, reflection_issues=[],
        reflection_brand_issues={}, reflection_increase_sample=[],
        reflection_iter=0,
        conversation_history=list(_conversation_history),
        viz_charts={}, chart_commentary={}, report_path=None,
        retrieval_metrics={}, efficiency_metrics={},
        messages=[], error=None, guidance_message=None,
        _df=df_clean, _start_time=time.time(),
    )
    return app.invoke(initial, config={"recursion_limit": 50})


LangGraph workflow compiled.
  coordinator -> cache_inject -> [search (if new)] -> analysis -> reflection -> viz -> report


In [17]:
# Cell P: Metrics evaluator + test suite

class MetricsEvaluator:
    """Compute quantitative metrics across retrieval, sentiment, and efficiency."""

    @staticmethod
    def retrieval(state: AgentState) -> Dict:
        target = {b.lower() for b in state.get("target_brands", [])}
        found  = {b.lower() for b in state.get("product_reviews", {})}
        total  = sum(len(df) for df in state.get("product_reviews", {}).values())
        return {
            "hit_rate":        len(target & found) / len(target) if target else 1.0,
            "brands_retrieved": len(found),
            "total_reviews":   total,
        }

    @staticmethod
    def sentiment(state: AgentState) -> Dict:
        llm_r = [r for r in state.get("sentiment_results", [])
                 if r.get("method") not in ("baseline", "baseline-fallback")]
        if not llm_r:
            return {}
        return {
            "avg_positive_pct":   float(np.mean([r["positive_pct"]   for r in llm_r])),
            "avg_confidence":     float(np.mean([r["avg_confidence"]  for r in llm_r])),
            "brands_with_aspects": sum(1 for r in llm_r if r.get("aspect_summary")),
            "reflection_score":   state.get("reflection_score"),
            "reflection_iter":    state.get("reflection_iter", 0),
        }

    @staticmethod
    def efficiency(state: AgentState) -> Dict:
        elapsed = time.time() - (state.get("_start_time") or time.time())
        reviews = sum(len(df) for df in state.get("product_reviews", {}).values())
        est_tokens = reviews * 130
        est_cost   = est_tokens * 0.15 / 1_000_000  # gpt-4o-mini input rate
        return {"elapsed_sec": round(elapsed, 1), "est_cost_usd": round(est_cost, 5)}

    @classmethod
    def report(cls, state: AgentState):
        ret  = cls.retrieval(state)
        sent = cls.sentiment(state)
        eff  = cls.efficiency(state)
        print("\n--- EVALUATION METRICS ---")
        print(f"Retrieval  | hit_rate={ret['hit_rate']:.0%}  brands={ret['brands_retrieved']}  "
              f"reviews={ret['total_reviews']}")
        if sent:
            print(f"Sentiment  | avg_positive={sent['avg_positive_pct']:.1f}%  "
                  f"avg_confidence={sent['avg_confidence']:.2f}  "
                  f"reflection={sent['reflection_score']}/10  "
                  f"aspects_covered={sent['brands_with_aspects']} brand(s)")
        print(f"Efficiency | {eff['elapsed_sec']}s  est_cost=${eff['est_cost_usd']}")


# ── Test suite ────────────────────────────────────────────────────────────────

TEST_CASES = [
    {"query": "Compare Samsung and Nokia on battery life",
     "expect_intent": "compare", "expect_brands": ["Samsung", "Nokia"]},
    {"query": "Analyze Apple iPhone reviews",
     "expect_intent": "analyze", "expect_brands": ["Apple"]},
    {"query": "Find phones with good camera",
     "expect_intent": "search",  "expect_brands": []},
]


def run_unit_tests():
    """Coordinator unit tests: verify intent parsing without running full pipeline."""
    print("=" * 60)
    print("UNIT TESTS — coordinator only (no sentiment API calls)")
    print("=" * 60)
    llm = ChatOpenAI(model=CONFIG["model"], temperature=0)
    passed = 0
    for i, tc in enumerate(TEST_CASES, 1):
        prompt = Prompts.COORDINATOR.format(query=tc["query"], few_shots="", conversation_ctx="", cached_brands="")
        try:
            result = llm.with_structured_output(QueryIntent).invoke([
                SystemMessage("You are a product review query parser."),
                HumanMessage(prompt),
            ])
            intent_ok = result.intent == tc["expect_intent"]
            brand_ok  = all(
                b.lower() in [x.lower() for x in result.brands]
                for b in tc["expect_brands"]
            )
            status = "PASS" if (intent_ok and brand_ok) else "FAIL"
            passed += int(status == "PASS")
            print(f"  [{status}] Test {i}: intent={result.intent}  brands={result.brands}")
        except Exception as e:
            print(f"  [ERROR] Test {i}: {e}")
    print(f"\n{passed}/{len(TEST_CASES)} unit tests passed.")


def run_integration_test():
    """Full pipeline integration test on test case 1."""
    print("=" * 60)
    print("INTEGRATION TEST — full pipeline")
    print("=" * 60)
    result = run_query(TEST_CASES[0]["query"])
    MetricsEvaluator.report(result)
    return result


run_unit_tests()

UNIT TESTS — coordinator only (no sentiment API calls)
  [PASS] Test 1: intent=compare  brands=['Samsung', 'Nokia']
  [PASS] Test 2: intent=analyze  brands=['Apple']
  [PASS] Test 3: intent=search  brands=[]

3/3 unit tests passed.


In [ ]:
# Cell Q: Gradio interface

# CSS for tables, history panel, and agent progress bar
_CSS = """
.results-table { border-collapse: collapse; width: 100%; font-size: 14px; }
.results-table th { background: #4472C4; color: #fff; padding: 8px 12px; text-align: left; }
.results-table td { padding: 6px 12px; border-bottom: 1px solid #ddd; }
.results-table tr:nth-child(even) { background: #f5f7ff; }
.history-table { border-collapse: collapse; width: 100%; font-size: 13px; margin-top: 8px; }
.history-table th { background: #5b9bd5; color: #fff; padding: 6px 10px; }
.history-table td { padding: 5px 10px; border-bottom: 1px solid #ddd; vertical-align: top; }
.agent-progress { display: flex; gap: 6px; flex-wrap: wrap; margin: 8px 0; font-size: 13px; }
.agent-step { padding: 4px 12px; border-radius: 12px; font-weight: 500; }
.step-done    { background: #d4edda; color: #155724; }
.step-active  { background: #fff3cd; color: #856404; }
.step-pending { background: #f0f0f0; color: #888; }
"""

# Agent pipeline steps in execution order
AGENT_STEPS = [
    ("coordinator",   "Coordinator"),
    ("cache_inject",  "Session Cache"),
    ("search",        "Review Search"),
    ("analysis",      "Sentiment Analysis"),
    ("reflection",    "Quality Check"),
    ("visualization", "Charts"),
    ("report",        "Report"),
]


def _progress_html(current_node, done_nodes):
    """Render a pill-row showing done / active / pending pipeline steps."""
    items = []
    for node, label in AGENT_STEPS:
        if node in done_nodes:
            cls, icon = "step-done", "&#10003;"
        elif node == current_node:
            cls, icon = "step-active", "&#8987;"
        else:
            cls, icon = "step-pending", "&#9675;"
        items.append(f'<span class="agent-step {cls}">{icon} {label}</span>')
    return '<div class="agent-progress">' + "".join(items) + "</div>"


def _table_html(df: pd.DataFrame) -> str:
    return df.to_html(index=False, border=0, classes="results-table")


def _history_html() -> str:
    if not _conversation_history:
        return "<p><em>No queries yet in this session.</em></p>"
    rows = ""
    for turn in _conversation_history:
        rows += (
            f"<tr>"
            f"<td>{turn['turn']}</td>"
            f"<td>{turn['query']}</td>"
            f"<td>{turn['intent']}</td>"
            f"<td>{', '.join(turn['brands'])}</td>"
            f"<td>{turn.get('summary', '')}</td>"
            f"</tr>"
        )
    return (
        '<table class="history-table">'
        "<thead><tr>"
        "<th>#</th><th>Query</th><th>Intent</th><th>Brands</th><th>Summary</th>"
        "</tr></thead>"
        f"<tbody>{rows}</tbody>"
        "</table>"
    )


def _format_outputs(state):
    """Format final AgentState into display HTML strings."""
    # Off-topic: coordinator set guidance_message -> short-circuit with a friendly guide
    guidance = state.get("guidance_message")
    if guidance:
        guide_html = (
            "<div style='padding:16px;background:#e8f4f8;border-left:5px solid #4472C4;border-radius:4px;'>"
            "<b>&#8505;&nbsp;This query is outside my scope.</b><br><br>"
            + guidance.replace("\n", "<br>")
            + "</div>"
        )
        return guide_html, "<p>-</p>", None, None, None, None, None, _history_html()

    score = state.get("reflection_score")
    quality_badge = (
        f'<p style="font-weight:bold">Analysis quality: {score}/10</p>' if score else ""
    )
    comp_records = state.get("comparison_table")
    # comparison_table is List[Dict] in state — convert to DataFrame for HTML rendering
    comp = pd.DataFrame(comp_records) if comp_records else None
    # Surface brands that had no reviews found
    not_found = [r for r in state.get("sentiment_results", []) if r.get("method") == "not_found"]
    not_found_html = ""
    if not_found:
        rows = "".join(
            f"<tr><td><b>{r['brand']}</b></td><td style='color:#8B1E28'>{r.get('error','No reviews found')}</td></tr>"
            for r in not_found
        )
        not_found_html = (
            "<div style='margin-top:10px;padding:10px;background:#fff3cd;border-left:4px solid #ffc107;'>"
            "<b>Brands with no results:</b>"
            f"<table class='results-table' style='margin-top:6px'><tr><th>Brand</th><th>Reason</th></tr>{rows}</table>"
            "</div>"
        )
    comp_html = quality_badge + (_table_html(comp) if comp is not None else "<p>No comparison data.</p>") + not_found_html

    asp_html = ""
    for brand, aspects in state.get("aspect_summary", {}).items():
        if aspects:
            asp_df = pd.DataFrame([
                {
                    "Aspect": a,
                    "Positive %": f"{v['positive']*100:.0f}%",
                    "Negative %": f"{v['negative']*100:.0f}%",
                    "Mentions": v["mentions"],
                }
                for a, v in sorted(aspects.items(), key=lambda x: -x[1]["mentions"])
            ])
            asp_html += f"<h3>{brand}</h3>" + _table_html(asp_df)
    if not asp_html:
        asp_html = "<p>No aspect data available.</p>"

    _charts = state.get("viz_charts", {})

    fig1 = _charts.get("sentiment_distribution")
    fig2 = _charts.get("aspect_heatmap")
    fig3 = _charts.get("diverging_bar")
    fig4 = _charts.get("brand_comparison")

    return comp_html, asp_html, fig1, fig2, fig3, fig4, state.get("report_path"), _history_html()


def gradio_query(query: str):
    """Stream pipeline progress then yield final results."""
    if not query.strip():
        empty = "<p>Enter a query above.</p>"
        # 增加一個 None 對應 fig4 的空缺
        yield empty, empty, empty, None, None, None, None, None, _history_html()
        return

    empty = "<p>Processing&#8230;</p>"
    done_nodes: set = set()
    step_names = [n for n, _ in AGENT_STEPS]

    initial = AgentState(
        query=query, intent="", target_brands=[], aspects=[],
        reuse_brands=[], new_brands=[],
        product_reviews={}, sentiment_results=[], aspect_summary={},
        comparison_table=None,
        reflection_score=None, reflection_issues=[],
        reflection_brand_issues={}, reflection_increase_sample=[],
        reflection_iter=0,
        conversation_history=list(_conversation_history),
        viz_charts={}, chart_commentary={}, report_path=None,
        retrieval_metrics={}, efficiency_metrics={},
        messages=[], error=None, guidance_message=None,
        _df=df_clean, _start_time=time.time(),
    )

    try:
        last_state = dict(initial)
        for chunk in app.stream(initial, config={"recursion_limit": 50}):
            node_name = next(iter(chunk))
            done_nodes.add(node_name)
            last_state.update(chunk[node_name])

            idx = step_names.index(node_name) if node_name in step_names else -1
            next_node = step_names[idx + 1] if 0 <= idx + 1 < len(step_names) else None

            # Surface charts immediately after visualization (don't wait for report)
            if node_name == "visualization":
                _vc = last_state.get("viz_charts", {})
                yield (
                    _progress_html(next_node, done_nodes), empty, empty,
                    _vc.get("sentiment_distribution"),
                    _vc.get("aspect_heatmap"),
                    _vc.get("diverging_bar"),
                    _vc.get("brand_comparison"),
                    None, _history_html()
                )
            else:
                yield _progress_html(next_node, done_nodes), empty, empty, None, None, None, None, None, _history_html()

                # Pipeline done — yield main results immediately
        comp_html, asp_html, fig1, fig2, fig3, fig4, _, hist_html = _format_outputs(last_state)
        yield _progress_html("report", done_nodes), comp_html, asp_html, fig1, fig2, fig3, fig4, None, hist_html

        # --- Print metrics to notebook output ---
        _ret = MetricsEvaluator.retrieval(last_state)
        _eff = MetricsEvaluator.efficiency(last_state)
        print(f"{'='*55}")
        print("QUERY METRICS")
        print(f"{'='*55}")
        print(f"Hit Rate : {_ret['hit_rate']:.0%}  |  Brands: {_ret['brands_retrieved']}  |  Reviews: {_ret['total_reviews']}")
        # print(f"Elapsed  : {_eff['elapsed_s']:.1f}s  |  Est. Cost: ${_eff['est_cost_usd']:.4f}  |  Reflection: {last_state.get('reflection_score', '-')}/10")
        _all_res = last_state.get('sentiment_results', [])
        _llm_map  = {r['brand']: r for r in _all_res if r.get('method') not in ('baseline', 'baseline-fallback', 'not_found')}
        _base_map = {r['brand']: r for r in _all_res if r.get('method') in ('baseline', 'baseline-fallback')}
        if _llm_map and _base_map:
            print(f"{'GPT-4o-mini vs TextBlob':^55}")
            print('-' * 55)
            print(f"{'Brand':<12} {'Method':<13} {'Pos%':>6} {'Neg%':>6} {'Polarity':>9} {'Conf':>6}")
            print('-' * 55)
            for _brand, _llm in _llm_map.items():
                _base = _base_map.get(_brand)
                print(f"{_brand:<12} {'GPT-4o-mini':<13} {_llm['positive_pct']:>5.1f}% {_llm['negative_pct']:>5.1f}% {_llm['avg_polarity']:>+9.3f} {_llm['avg_confidence']:>6.3f}")
                if _base:
                    print(f"{'':12} {'TextBlob':<13} {_base['positive_pct']:>5.1f}% {_base['negative_pct']:>5.1f}% {_base['avg_polarity']:>+9.3f} {'—':>6}")
            print('-' * 55)

        # Generate report with chart images in background thread (won't block UI)
        import threading as _threading
        _report_result = [None]
        _report_event = _threading.Event()

        def _bg_report():
            try:
                _report_result[0] = report_generator(dict(last_state)).get("report_path")
            except Exception as _re:
                print(f"  Background report error: {_re}")
            finally:
                _report_event.set()

        _threading.Thread(target=_bg_report, daemon=True).start()
        _report_event.wait(timeout=120)  # wait up to 2 min; user already has results
        done_nodes.add("report")
        done_nodes.update(n for n, _ in AGENT_STEPS)


        yield _progress_html(None, done_nodes), comp_html, asp_html, fig1, fig2, fig3, fig4, _report_result[0], hist_html


    except Exception as e:
        import traceback as _tb
        from langgraph.errors import GraphRecursionError
        if isinstance(e, GraphRecursionError):
            msg = (
                "<div style='padding:12px;background:#fff3cd;border-left:5px solid #ffc107;border-radius:4px;'>"
                "<b>&#9888;&nbsp;No relevant information found.</b><br><br>"
                "The system could not find enough review data for this query after multiple attempts. "
                "Please try different brand names or a simpler query.</div>"
            )
            yield msg, msg, "<p>-</p>", None, None, None, None, None, _history_html()
        else:
            err = f"<p>Pipeline error: {e}</p><pre>{_tb.format_exc()}</pre>"
            yield err, err, err, None, None, None, None, None, _history_html()


def gradio_clear_session():
    """Wipe session cache and return a reset state."""
    clear_session()
    cleared = "<p>Session cleared - all cached brand reviews and conversation history removed.</p>"
    return (
        "<p><em>Session cleared.</em></p>",
        cleared,
        "<p>-</p>",
        None,
        None,
        None,
        None,
        None,
        "<p><em>Session cleared.</em></p>",
    )

custom_theme = gr.themes.Soft(
    font=["Arial", "Helvetica", "ui-sans-serif", "system-ui", "sans-serif"],
    text_size=gr.themes.sizes.text_lg
)
with gr.Blocks(theme=custom_theme, css=_CSS) as demo:
    gr.Markdown(
        "## AI Product Review Intelligence System\n"
        "*Hybrid RAG (BM25 + Bi-encoder + Cross-encoder) \u00b7 "
        "ReAct Agent \u00b7 Aspect-Level CoT Sentiment \u00b7 "
        "Reflection (Evaluator-Optimizer) \u00b7 Episodic Memory \u00b7 Multi-Turn Session Cache*"
    )

    with gr.Row():
        query_box = gr.Textbox(
            label="Query",
            placeholder="e.g., Compare Samsung and Nokia on battery",
            lines=2, scale=5,
        )
        with gr.Column(scale=1):
            submit_btn = gr.Button("Analyze", variant="primary")
            clear_btn  = gr.Button("Clear Session", variant="secondary")

    gr.Examples(
        examples=[
            ["Compare Samsung and Nokia on battery life"],
            ["Now also add Apple to the comparison"],
            ["Analyze Apple iPhone reviews"],
            ["Find phones with good camera"],
            ["Compare Samsung, LG, and BLU on performance and price"],
        ],
        inputs=query_box,
    )

    # Pipeline progress bar - updates in real-time during analysis
    out_status = gr.HTML(label="Pipeline Status")

    with gr.Tabs():
        with gr.Tab("Comparison & Quality"):
            out_comp = gr.HTML()
        with gr.Tab("Aspect Breakdown"):
            out_asp = gr.HTML()
        with gr.Tab("Charts"):
            out_chart1 = gr.Plot(label="Overall Sentiment Distribution")
            out_chart2 = gr.Plot(label="Aspect Satisfaction Heatmap")
            out_chart3 = gr.Plot(label="Sentiment Polarization (Diverging)")
            out_chart4 = gr.Plot(label="Volume vs. Satisfaction Matrix")
        with gr.Tab("Report"):
            out_report = gr.File(label="Download .docx")
        with gr.Tab("Conversation History"):
            gr.Markdown(
                "All queries in this session \u2014 brands served from cache skip re-fetching "

            )
            out_history = gr.HTML()

    outputs = [out_status, out_comp, out_asp, out_chart1, out_chart2, out_chart3, out_chart4, out_report, out_history]

    for trigger in [submit_btn.click, query_box.submit]:
        trigger(fn=gradio_query, inputs=[query_box], outputs=outputs)

    clear_btn.click(fn=gradio_clear_session, inputs=[], outputs=outputs)


import nest_asyncio as _nest_asyncio
_nest_asyncio.apply()

if IN_COLAB:
    import asyncio as _asyncio
    import uvicorn.server as _uvi_server

    def _patched_server_run(self, sockets=None):
        loop = _asyncio.new_event_loop()
        _asyncio.set_event_loop(loop)
        try:
            loop.run_until_complete(self.serve(sockets=sockets))
        finally:
            loop.close()
            _asyncio.set_event_loop(None)

    _uvi_server.Server.run = _patched_server_run

demo.launch(share=True, debug=True)


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://0c09edcccc9fca51ce.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)



COORDINATOR
Query: Compare Samsung and Nokia on battery life
  Intent: compare  |  Brands: ['Samsung', 'Nokia']
  Reuse (cached): []  |  New (search): ['Samsung', 'Nokia']

CACHE INJECT AGENT

REACT SEARCH AGENT
  Cached 30 reviews for: Samsung
  Cached 30 reviews for: Nokia
  Total product_reviews: 2 brand(s), 60 reviews

ANALYSIS AGENT
Analyzing 2 brand(s) in parallel (retry=False, temp=0.0)...
Done in 32.7s
  Samsung: 33% positive | conf=0.88 | aspects=['build_quality', 'interface', 'battery', 'photo quality', 'performance', 'overall_experience', 'battery life', 'overall experience', 'screen', 'customer_service', 'delivery', 'customer service', 'SIM card requirement', 'music player', 'adapter_quality', 'features', 'software']
  Nokia: 20% positive | conf=0.88 | aspects=['battery', 'music functionality', 'Ovi calendar', 'apps', 'customer_service', 'reliability', 'camera', 'performance', 'functionality', 'build_quality', 'seller_service', 'network connection', 'sim card replacement',